# Job Application Tracker

Copy a job link from **LinkedIn** or **Handshake**, run one cell, confirm the fields it
pulled, and the row lands inside the `JobApplications` table in Google Sheets.

**How to use it**
1. Run cells 1-7 once per session (they only define things).
2. Copy the job URL to your clipboard.
3. Run `add_job()` in the last cell. Press **Enter** to accept any value it guessed, or type a correction.

**How it reads each site**
- **LinkedIn** is read from the public *guest* job card plus the page's `JobPosting` structured
  data (schema.org JSON-LD), so Company does not swallow the location
  (`Anker Innovations - Greater Seattle Area` becomes `Anker Innovations`).
- **Handshake** job pages sit behind a student login, so no scraper can read them. When the page
  cannot be read, the notebook asks you to paste the copied header block and parses that instead.

**What it normalises before writing**
- `Type` is one of Full-time / Part-time / Remote / Internship. It does not match *intern* inside
  *international*, and does not flag a posting Remote just because the word appears somewhere.
- `Salary` is captured with its correct period (`/hr`, `/mo`, `/yr`).
- `Job URL` is stripped of tracking parameters.
- Duplicate URLs are flagged before anything is written.

> **Setup required.** This notebook needs *your own* Google service-account key saved as
> `credentials.json` in this same folder, and your own spreadsheet shared with that service
> account. See the README for the full setup. No credentials are included in this repository.

### Notes

**Handshake.** Job postings are only visible to signed-in students, so `requests` receives the login
page no matter what headers it sends. The paste fallback is the reliable path: open the posting,
select the block with the title, employer, location, pay and job type, copy it, and paste it at
the prompt. It parses those five fields in any order.

**Type vs. Location.** An internship that is also remote is saved as `Internship` with Location
`N/A`. To make Remote win instead, swap the two checks at the top of `finalize()`.

**Status column.** If Status is a dropdown in your sheet, `"Applied"` has to match one of the
allowed values exactly or the cell shows a validation warning.

**Clipboard on Linux.** `pyperclip` needs `xclip` or `xsel` installed. Without it the notebook
simply asks you to paste the URL.

**If a row lands below the table** instead of inside it, the table's range stopped short. Click any
cell in the table, drag the handle at the bottom-right down a few rows, and it will absorb new rows again.

## 1. Imports and settings

In [ ]:
import json
import re
from datetime import datetime
from urllib.parse import urlparse, parse_qs

import requests
from bs4 import BeautifulSoup

try:
    import pytz
except ImportError:
    pytz = None

try:
    import gspread
    from google.oauth2.service_account import Credentials
except ImportError:
    gspread = None

try:
    import pyperclip
except Exception:          # clipboard is optional (needs xclip on Linux)
    pyperclip = None


# ------------------------------- CONFIG -------------------------------
SHEET_NAME       = "Job_Applications"    # spreadsheet file name in Drive
WORKSHEET        = None                  # None = first tab, or e.g. "Sheet1"
CREDENTIALS_FILE = "credentials.json"
TIMEZONE         = "US/Pacific"
DATE_FORMAT      = "%d-%b"               # 22-Aug
DEFAULT_STATUS   = "Applied"
CHECK_DUPLICATES = True                  # warn if the Job URL is already logged

HEADERS = [
    "Position", "Company", "Type", "Location",
    "Salary", "Website", "Job URL", "Date Applied", "Status",
]

TYPE_CHOICES = ["Full-time", "Part-time", "Remote", "Internship"]

BROWSER_HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) "
                   "Chrome/124.0.0.0 Safari/537.36"),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

print("Config loaded.")

## 2. Text and URL helpers

In [ ]:
def clean_text(value):
    """Collapse whitespace, normalise dashes, return a plain string."""
    if value is None:
        return ""
    text = str(value)
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def today_str():
    if pytz:
        now = datetime.now(pytz.timezone(TIMEZONE))
    else:
        now = datetime.now()
    return now.strftime(DATE_FORMAT)


def detect_site(url):
    """Only LinkedIn or Handshake are valid values for the Website column."""
    host = urlparse(url).netloc.lower()
    if "linkedin." in host:
        return "LinkedIn"
    if "joinhandshake." in host or "handshake." in host:
        return "Handshake"
    return ""


def linkedin_job_id(url):
    """Works for /jobs/view/4435327051/ and /jobs/view/some-title-4435327051."""
    m = re.search(r"/jobs/view/(?:[^/?#]*?-)?(\d{6,})", url)
    if m:
        return m.group(1)
    query = parse_qs(urlparse(url).query)
    for key in ("currentJobId", "jobId", "jobPostingId"):
        if query.get(key) and query[key][0].isdigit():
            return query[key][0]
    return ""


def handshake_job_id(url):
    m = re.search(r"/(?:stu/)?(?:jobs|job-search|postings)/(\d{3,})", url)
    if m:
        return m.group(1)
    query = parse_qs(urlparse(url).query)
    for key in ("job_id", "jobId", "id"):
        if query.get(key) and query[key][0].isdigit():
            return query[key][0]
    return ""


def canonical_url(url):
    """Strip the tracking junk so the sheet holds a short, clickable link."""
    url = url.strip()
    parsed = urlparse(url)
    site = detect_site(url)

    if site == "LinkedIn":
        job_id = linkedin_job_id(url)
        if job_id:
            return f"https://www.linkedin.com/jobs/view/{job_id}/"
    if site == "Handshake":
        job_id = handshake_job_id(url)
        host = parsed.netloc or "app.joinhandshake.com"
        if job_id:
            return f"https://{host}/jobs/{job_id}"

    if parsed.scheme and parsed.netloc:
        return f"{parsed.scheme}://{parsed.netloc}{parsed.path}".rstrip("/") or url
    return url


def new_record(url):
    return {
        "Position": "", "Company": "", "Type": "", "Location": "", "Salary": "",
        "Website": detect_site(url),
        "Job URL": canonical_url(url),
        "Date Applied": today_str(),
        "Status": DEFAULT_STATUS,
        "_remote": None,      # True / False / None (unknown)
        "_employment": "",    # raw employment-type string from the page
        "_text": "",          # trimmed page text used for salary/type sniffing
        "_scraped": False,    # did we get anything useful?
    }


def fill_blanks(record, found):
    """First source wins: only write into fields that are still empty."""
    for key, value in (found or {}).items():
        if key in ("_remote",):
            if record.get("_remote") is None and value is not None:
                record["_remote"] = value
        elif key in ("_text", "_employment"):
            if value and not record.get(key):
                record[key] = value
        elif value and not record.get(key):
            record[key] = value
            record["_scraped"] = True
    return record


print("Helpers loaded.")

## 3. Salary, type and location rules

In [ ]:
PERIOD_SUFFIX = {"H": "/hr", "D": "/day", "W": "/wk", "M": "/mo", "Y": "/yr"}

PERIOD_WORDS = [
    (r"(per\s*hour|an\s*hour|hourly|/\s*hour|/\s*hr\b|\bhr\b|\bhour\b)", "H"),
    (r"(per\s*day|daily|/\s*day\b|\bday\b)", "D"),
    (r"(per\s*week|a\s*week|weekly|/\s*week|/\s*wk\b|\bwk\b|\bweek\b)", "W"),
    (r"(per\s*month|a\s*month|monthly|/\s*month|/\s*mo\b|\bmo\b|\bmonth\b)", "M"),
    (r"(per\s*year|a\s*year|annually|annual|per\s*annum|/\s*year|/\s*yr\b|\byr\b|\byear\b)", "Y"),
]

AMOUNT_RE = re.compile(r"(\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?)(?:\s*([kKmMbB])\b)?")


def detect_period(text):
    text = (text or "").lower()
    for pattern, code in PERIOD_WORDS:
        if re.search(pattern, text):
            return code
    return ""


def infer_period(amount):
    """No period stated on the posting: guess from the size of the number."""
    if amount < 250:
        return "H"
    if amount < 20000:
        return "M"
    return "Y"


def format_salary(amounts, period):
    values = []
    for amount in amounts:
        try:
            value = float(str(amount).replace(",", "").replace("$", ""))
        except (TypeError, ValueError):
            continue
        if value > 0:
            values.append(value)
    if not values:
        return ""
    values = sorted(set(values))[:2]
    if not period:
        period = infer_period(values[0])
    body = " - ".join(f"${v:,.2f}" for v in values)
    return body + PERIOD_SUFFIX.get(period, "")


def parse_salary_string(text, require_dollar=False):
    """Turn '25/hr', '$85k - $110k a year', '1500 monthly' into a clean string."""
    text = clean_text(text)
    if not text:
        return ""
    if require_dollar and "$" not in text:
        return ""

    amounts = []
    for raw, suffix in AMOUNT_RE.findall(text.replace("$", " $ ")):
        if suffix.lower() in ("m", "b"):
            continue
        value = float(raw.replace(",", ""))
        if suffix:
            value *= 1000
        # ignore obvious non-money numbers (years, counts)
        if 1900 <= value <= 2100 and "," not in raw and not suffix:
            continue
        amounts.append(value)
    if not amounts:
        return ""
    return format_salary(amounts[:2], detect_period(text))


def find_salary_in_text(text):
    """Scan page text for a '$...' snippet that also states a pay period."""
    text = clean_text(text)
    for match in re.finditer(r"\$\s?\d", text):
        snippet = text[match.start(): match.start() + 70]
        if not detect_period(snippet):
            continue
        cut = re.split(r"\s{2,}|[|•·]", snippet)[0]
        salary = parse_salary_string(cut, require_dollar=True)
        if salary:
            return salary
    return ""


REMOTE_STRONG = re.compile(
    r"(fully|100%|entirely)\s+remote|\bremote\s+(role|position|job|opportunity|work)\b|"
    r"\bthis (role|position) is remote\b|work from home", re.I)


def looks_remote_in_text(text):
    """Long descriptions mention 'remote' casually, so require a strong phrase."""
    text = clean_text(text)
    if not text:
        return None
    if re.search(r"\bhybrid\b|\bon[- ]?site\b", text[:400], flags=re.I):
        return False
    return True if REMOTE_STRONG.search(text) else None


def looks_remote(*sources):
    text = " ".join(clean_text(s).lower() for s in sources if s)
    if not text:
        return None
    if re.search(r"\bhybrid\b|\bon[- ]?site\b|\bin[- ]?person\b", text):
        return False
    if re.search(r"\bremote\b|work from home|telecommute|\bwfh\b", text):
        return True
    return None


def normalize_type(*sources):
    text = " ".join(clean_text(s).lower() for s in sources if s)
    if re.search(r"\bintern(ship|s)?\b|\bco[- ]?op\b", text):
        return "Internship"
    if re.search(r"\bpart[\s-]?time\b", text):
        return "Part-time"
    if re.search(r"\bfull[\s-]?time\b|\bfull_time\b", text):
        return "Full-time"
    if re.search(r"\bcontract(or)?\b|\btemporary\b", text):
        return "Full-time"
    return ""


def normalize_location(text):
    text = clean_text(text)
    if not text:
        return ""
    text = re.sub(r"\((remote|hybrid|on[- ]?site|in[- ]?person)\)", "", text, flags=re.I)
    text = re.sub(r"[,\-–—]\s*(remote|hybrid|on[- ]?site)\s*$", "", text, flags=re.I)
    text = re.sub(r"\b(greater|metropolitan)\b", "", text, flags=re.I)
    text = re.sub(r"\bmetropolitan area\b|\barea\b", "", text, flags=re.I)
    text = re.sub(r"\s*,\s*", ", ", text).strip(" ,-–—")
    # "Seattle, Washington, United States" -> "Seattle, Washington"
    parts = [p for p in text.split(", ") if p]
    if len(parts) >= 3 and parts[-1].lower() in ("united states", "usa", "us"):
        parts = parts[:-1]
    return ", ".join(parts)


print("Parsers loaded.")

## 4. Reading the job page

In [ ]:
def fetch_soup(url, timeout=12):
    try:
        response = requests.get(url, headers=BROWSER_HEADERS, timeout=timeout)
        if response.status_code >= 400 or not response.text.strip():
            return None
        return BeautifulSoup(response.text, "html.parser")
    except Exception as error:
        print(f"  (could not load {url[:60]}... -> {error})")
        return None


def looks_like_login_wall(soup):
    title = clean_text(soup.title.get_text()) if soup and soup.title else ""
    body = clean_text(soup.get_text(" "))[:400].lower() if soup else ""
    markers = ("sign in", "log in", "sign up", "authwall", "join handshake",
               "welcome to handshake", "please log in")
    return any(m in title.lower() for m in markers) or any(m in body for m in markers)


def _jsonld_objects(soup):
    for tag in soup.find_all("script", attrs={"type": "application/ld+json"}):
        raw = tag.string or tag.get_text() or ""
        try:
            data = json.loads(raw)
        except (ValueError, TypeError):
            continue
        stack = [data]
        while stack:
            item = stack.pop()
            if isinstance(item, list):
                stack.extend(item)
            elif isinstance(item, dict):
                graph = item.get("@graph")
                if isinstance(graph, list):
                    stack.extend(graph)
                yield item


def _jsonld_location(node):
    nodes = node if isinstance(node, list) else [node]
    for item in nodes:
        if not isinstance(item, dict):
            continue
        address = item.get("address")
        if isinstance(address, list):
            address = address[0] if address else None
        if isinstance(address, dict):
            city = clean_text(address.get("addressLocality"))
            region = clean_text(address.get("addressRegion"))
            country = clean_text(address.get("addressCountry")) if not (city or region) else ""
            parts = [p for p in (city, region) if p] or [country]
            if any(parts):
                return ", ".join(parts)
        if clean_text(item.get("name")):
            return clean_text(item.get("name"))
    return ""


def _jsonld_salary(node):
    if not isinstance(node, dict):
        return ""
    value = node.get("value")
    if isinstance(value, list):
        value = value[0] if value else None

    period, amounts = "", []
    if isinstance(value, dict):
        period = {"HOUR": "H", "DAY": "D", "WEEK": "W", "MONTH": "M", "YEAR": "Y"}.get(
            str(value.get("unitText", "")).upper(), "")
        amounts = [value.get("minValue"), value.get("maxValue"), value.get("value")]
    elif value is not None:
        amounts = [value]
    amounts = [a for a in amounts if a not in (None, "", 0, "0")]
    return format_salary(amounts, period)


def parse_jsonld(soup):
    """Schema.org JobPosting — the cleanest source when a site publishes it."""
    for item in _jsonld_objects(soup):
        types = item.get("@type")
        types = types if isinstance(types, list) else [types]
        if "JobPosting" not in types:
            continue

        org = item.get("hiringOrganization")
        if isinstance(org, dict):
            company = clean_text(org.get("name"))
        else:
            company = clean_text(org)

        employment = item.get("employmentType")
        if isinstance(employment, list):
            employment = " ".join(str(e) for e in employment)

        description = BeautifulSoup(str(item.get("description", "")), "html.parser").get_text(" ")

        return {
            "Position": clean_text(item.get("title")),
            "Company": company,
            "Location": normalize_location(_jsonld_location(item.get("jobLocation"))),
            "Salary": _jsonld_salary(item.get("baseSalary")),
            "_employment": clean_text(employment).replace("_", "-"),
            "_remote": True if str(item.get("jobLocationType", "")).upper() == "TELECOMMUTE" else None,
            "_text": clean_text(description)[:4000],
        }
    return {}


def _first_text(soup, selectors):
    for selector in selectors:
        element = soup.select_one(selector)
        if element:
            text = clean_text(element.get_text(" "))
            if text:
                return text
    return ""


def parse_linkedin_topcard(soup):
    """LinkedIn's public 'guest' job card — reliable structured fields."""
    found = {
        "Position": _first_text(soup, [
            "h1.top-card-layout__title", "h2.top-card-layout__title",
            "h1.topcard__title", "h2.topcard__title", ".top-card-layout__title", "h1"]),
        "Company": _first_text(soup, [
            "a.topcard__org-name-link", ".topcard__org-name-link",
            ".top-card-layout__second-subline a", "span.topcard__flavor a",
            ".topcard__flavor--black-link"]),
        "Location": normalize_location(_first_text(soup, [
            ".topcard__flavor--bullet", ".top-card-layout__second-subline .topcard__flavor--bullet"])),
        "Salary": parse_salary_string(_first_text(soup, [
            ".compensation__salary", ".salary.compensation__salary", ".job-details-jobs-unified-top-card__job-insight"]),
            require_dollar=True),
    }

    criteria = []
    for item in soup.select("li.description__job-criteria-item, .description__job-criteria-item"):
        label = _first_text(item, [".description__job-criteria-subheader", "h3"])
        value = _first_text(item, [".description__job-criteria-text", "span"])
        if label and value:
            criteria.append(f"{label}: {value}")
            if "employment" in label.lower():
                found["_employment"] = value
            if "workplace" in label.lower():
                found["_remote"] = looks_remote(value)

    description = _first_text(soup, [
        ".description__text", ".show-more-less-html__markup", ".jobs-description__content"])
    found["_text"] = clean_text(" ".join(criteria + [description]))[:4000]
    return {k: v for k, v in found.items() if v not in ("", None)}


def split_title_blob(raw, site=""):
    """'Anker Innovations hiring GTM Specialist in Greater Seattle Area | LinkedIn'
       'GTM Specialist - Anker Innovations - Greater Seattle Area'
       'GTM Specialist at Anker Innovations | LinkedIn'"""
    raw = clean_text(raw)
    raw = re.sub(r"\s*[|\-–—]\s*(LinkedIn|Handshake|Indeed)\s*$", "", raw, flags=re.I)
    if not raw:
        return "", "", ""

    m = re.match(r"^(?P<company>.+?)\s+hiring\s+(?P<title>.+?)(?:\s+in\s+(?P<loc>.+))?$", raw, flags=re.I)
    if m:
        return clean_text(m.group("title")), clean_text(m.group("company")), clean_text(m.group("loc") or "")

    # " at " wins if present ("Analyst - Corporate Finance at Boeing"),
    # otherwise use whichever dash/pipe appears first.
    title, rest = raw, ""
    chosen = next((s for s in (" at ", " @ ") if s in raw), "")
    if not chosen:
        hits = [(raw.find(s), s) for s in (" — ", " – ", " - ", " | ") if s in raw]
        chosen = min(hits)[1] if hits else ""
    if chosen:
        title, rest = raw.split(chosen, 1)

    company, location = clean_text(rest), ""
    for separator in (" — ", " – ", " - ", " | ", ", "):
        if separator in company:
            head, tail = company.split(separator, 1)
            if re.search(r"\b(area|remote|metro|county|,)\b", tail, flags=re.I) or re.match(
                    r"^[A-Z][\w.'-]*(?:\s[\w.'-]+)*,\s*[A-Z]", tail):
                company, location = clean_text(head), clean_text(tail)
                break
    return clean_text(title), company, location


def parse_meta(soup, site=""):
    """Fallback: Open Graph / <title> tags."""
    def meta(prop, attr="property"):
        tag = soup.find("meta", attrs={attr: prop})
        return clean_text(tag.get("content")) if tag and tag.get("content") else ""

    raw_title = meta("og:title") or meta("twitter:title", "name") or (
        clean_text(soup.title.get_text()) if soup.title else "")
    description = meta("og:description") or meta("description", "name")

    title, company, location = split_title_blob(raw_title, site)
    return {
        "Position": title,
        "Company": company,
        "Location": normalize_location(location),
        "Salary": find_salary_in_text(description),
        "_text": clean_text(description)[:2000],
    }


def scrape_job(url):
    """Try every source we have, best-quality first, and fill the blanks."""
    record = new_record(url)
    site = record["Website"]
    pages = []

    if site == "LinkedIn":
        job_id = linkedin_job_id(url)
        if job_id:
            pages.append(f"https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/{job_id}")
        pages.append(record["Job URL"])
    else:
        pages.append(record["Job URL"])

    for page in pages:
        soup = fetch_soup(page)
        if not soup:
            continue
        if looks_like_login_wall(soup):
            print("  Page is behind a login wall — no data could be read from it.")
            continue
        fill_blanks(record, parse_jsonld(soup))
        if site == "LinkedIn":
            fill_blanks(record, parse_linkedin_topcard(soup))
        fill_blanks(record, parse_meta(soup, site))
        if record["Position"] and record["Company"]:
            break

    finalize(record)
    return record


def finalize(record):
    """Apply the Type / Location / Salary rules once all sources are merged."""
    if record["_remote"] is None:
        record["_remote"] = looks_remote(record["Location"], record["_employment"],
                                         record["Position"])
    if record["_remote"] is None:
        record["_remote"] = looks_remote_in_text(record["_text"][:1500])

    job_type = normalize_type(record["_employment"], record["Position"], record["_text"][:600])
    if not job_type:
        job_type = "Remote" if record["_remote"] else "Full-time"
    if job_type == "Full-time" and record["_remote"]:
        job_type = "Remote"
    record["Type"] = job_type

    if record["_remote"] or record["Type"] == "Remote":
        record["Location"] = "N/A"
    else:
        record["Location"] = normalize_location(record["Location"])

    if not record["Salary"] and record["_text"]:
        record["Salary"] = find_salary_in_text(record["_text"])
    return record


print("Scrapers loaded.")

## 5. Paste fallback (Handshake)

In [ ]:
NOISE_LINES = re.compile(
    r"^(apply|save|share|report|job|jobs|posted|expires|applicants?|about the (job|role)|"
    r"qualifications|easy apply|show more|back to jobs|\d+ applicants?|posted \d+)",
    re.I)

CITY_STATE = re.compile(r"^[A-Z][\w.'’-]*(?:[ \-][\w.'’-]+)*,\s*(?:[A-Z]{2}|[A-Z][a-z]+)\.?$")

TYPE_TOKEN = re.compile(
    r"^(full[\s-]?time|part[\s-]?time|internship|intern|co[\s-]?op|contract(or)?|"
    r"temporary|seasonal|volunteer|apprenticeship|fellowship|permanent)$", re.I)

PLACE_TOKEN = re.compile(r"^(remote|hybrid|on[\s-]?site|in[\s-]?person|flexible)$", re.I)


def is_type_line(line):
    """True only if the WHOLE line is job-type keywords ('Internship',
    'Full-Time · Remote') — so a title like 'Business Analyst Intern' is safe."""
    tokens = [t.strip() for t in re.split(r"[·|,/•]| - ", line) if t.strip()]
    if not tokens or len(tokens) > 3:
        return False
    return all(TYPE_TOKEN.match(t) or PLACE_TOKEN.match(t) for t in tokens)


def read_block(prompt):
    """Read several pasted lines; a blank line ends the block."""
    print(prompt)
    lines = []
    while True:
        try:
            line = input()
        except EOFError:
            break
        if not line.strip():
            break
        lines.append(line)
    return "\n".join(lines)


def parse_pasted_block(text):
    """Pull fields out of text copied straight off a job page."""
    found = {"Position": "", "Company": "", "Location": "", "Salary": "",
             "_employment": "", "_remote": None, "_text": clean_text(text)[:4000]}
    if not clean_text(text):
        return {}

    leftovers = []
    for line in text.splitlines():
        line = clean_text(line)
        if not line or NOISE_LINES.match(line):
            continue
        if not found["Salary"] and "$" in line:
            salary = parse_salary_string(line, require_dollar=True)
            if salary:
                found["Salary"] = salary
                continue
        if is_type_line(line):
            if not found["_employment"] and normalize_type(line):
                found["_employment"] = line
            if found["_remote"] is None and looks_remote(line) is not None:
                found["_remote"] = looks_remote(line)
            continue
        if not found["Location"] and (CITY_STATE.match(line) or re.search(r"\bremote\b", line, flags=re.I)):
            found["Location"] = normalize_location(line)
            if found["_remote"] is None:
                found["_remote"] = looks_remote(line)
            continue
        leftovers.append(line)

    if leftovers:
        found["Position"] = leftovers[0]
    if len(leftovers) > 1:
        found["Company"] = leftovers[1]
    return {k: v for k, v in found.items() if v not in ("", None)}


print("Paste fallback loaded.")

## 6. Google Sheets

In [ ]:
def connect_to_sheet():
    scopes = ["https://www.googleapis.com/auth/spreadsheets",
              "https://www.googleapis.com/auth/drive"]
    creds = Credentials.from_service_account_file(CREDENTIALS_FILE, scopes=scopes)
    client = gspread.authorize(creds)

    try:
        spreadsheet = client.open(SHEET_NAME)
    except gspread.exceptions.SpreadsheetNotFound:
        print(f"Sheet '{SHEET_NAME}' not found. Creating a new one...")
        spreadsheet = client.create(SHEET_NAME)
        email = input("Enter your personal Gmail to share the new sheet with: ").strip()
        if email:
            spreadsheet.share(email, perm_type="user", role="writer")
        spreadsheet.sheet1.append_row(HEADERS)

    return spreadsheet.worksheet(WORKSHEET) if WORKSHEET else spreadsheet.sheet1


def next_data_row(worksheet):
    """Last filled row across Position (A) and Job URL (G), +1."""
    lengths = [len(worksheet.col_values(1)), len(worksheet.col_values(7))]
    return max(max(lengths) + 1, 2)


def already_logged(worksheet, url):
    key = url.rstrip("/").lower()
    for value in worksheet.col_values(7)[1:]:
        if clean_text(value).rstrip("/").lower() == key:
            return True
    return False


def write_row(worksheet, record):
    row = [record[column] for column in HEADERS]
    target = next_data_row(worksheet)
    # Selecting the exact A:I block keeps the row inside the JobApplications table.
    cells = worksheet.range(f"A{target}:I{target}")
    for cell, value in zip(cells, row):
        cell.value = str(value)
    worksheet.update_cells(cells, value_input_option="RAW")
    return target


print("Sheets functions loaded.")

## 7. Review and save

In [ ]:
def ask(label, default=""):
    answer = input(f"{label} [{default}]: " if default else f"{label}: ").strip()
    return answer or default


def ask_type(default):
    print("  1) Full-time   2) Part-time   3) Remote   4) Internship")
    answer = ask("Type", default)
    if answer in ("1", "2", "3", "4"):
        return TYPE_CHOICES[int(answer) - 1]
    key = answer.lower().replace("-", "").replace(" ", "")
    for choice in TYPE_CHOICES:
        if choice.lower().replace("-", "") == key:
            return choice
    return answer


def ask_salary(default):
    answer = ask("Salary (e.g. 25/hr, 1500/mo, 85k-110k/yr)", default)
    if not answer or answer == default:
        return answer
    parsed = parse_salary_string(answer)
    if parsed and not detect_period(answer):
        period = input("Is this per (H)our, (M)onth, or (Y)ear? [H/M/Y]: ").strip().upper()
        if period in PERIOD_SUFFIX:
            amounts = re.findall(r"[\d,]+(?:\.\d+)?", answer.replace("$", ""))
            values = [float(a.replace(",", "")) * (1000 if "k" in answer.lower() else 1) for a in amounts]
            return format_salary(values[:2], period)
    return parsed or answer


def add_job(url=None):
    print("Connecting to Google Sheets...")
    worksheet = connect_to_sheet()

    if not url:
        clipboard = ""
        if pyperclip:
            try:
                clipboard = pyperclip.paste().strip()
            except Exception:
                clipboard = ""
        if clipboard.startswith("http"):
            url = clipboard
            print(f"\nDetected URL from clipboard: {url[:90]}")
        else:
            url = input("\nNo URL in clipboard. Paste the job URL: ").strip()

    if not url.startswith("http"):
        print("That doesn't look like a URL. Stopping.")
        return

    print("\nReading the job posting...")
    record = scrape_job(url)

    if CHECK_DUPLICATES and already_logged(worksheet, record["Job URL"]):
        if ask("This job URL is already in the sheet. Add it anyway? (y/n)", "n").lower() != "y":
            print("Cancelled.")
            return

    if not record["_scraped"] or not record["Position"]:
        print("\nCouldn't read much from the page (Handshake pages require a login).")
        print("Open the posting, select the header block (title, company, location,")
        print("pay, job type), copy it, and paste it below — or just press Enter to")
        print("type the fields yourself.")
        pasted = read_block("\nPaste here, then press Enter on an empty line:")
        if pasted.strip():
            fill_blanks(record, parse_pasted_block(pasted))
            finalize(record)

    if not record["Website"]:
        record["Website"] = ask("Website (LinkedIn or Handshake)", "LinkedIn")

    print("\n--- Review Data Before Sending to Google Sheets ---")
    record["Position"] = ask("Position", record["Position"])
    record["Company"] = ask("Company", record["Company"])
    record["Type"] = ask_type(record["Type"] or "Full-time")

    if record["Type"].strip().lower() == "remote":
        record["Location"] = "N/A"
        print("Location automatically set to N/A for a remote role.")
    else:
        default_location = "" if record["Location"] == "N/A" else record["Location"]
        record["Location"] = ask("Location (e.g. Seattle, WA)", default_location)

    record["Salary"] = ask_salary(record["Salary"])
    record["Date Applied"] = ask("Date Applied", record["Date Applied"])
    record["Status"] = ask("Status", record["Status"])

    target = write_row(worksheet, record)
    print(f"\nSuccess! '{record['Position']}' inserted inside your table at row {target}.")
    return record


print("Ready. Run add_job() in the next cell.")

## 8. Add a job

In [ ]:
# Copy a LinkedIn or Handshake job link, then run this cell.
# You can also pass the link directly:  add_job("https://...")

add_job()